In [ ]:
!pip install torchsummary
!pip install torchinfo

## Tiny Yolo

In [1]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimov2

    # Get the YOLO model
    model = tinysimov2.yolo_v8_s()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


## V3 Fused weight

### Before fusuion


In [1]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimov3

    # Get the YOLO model
    model = tinysimov3.yolo_v8_s()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


### After fusion

In [ ]:
import torch
from torchsummary import summary
from contextlib import redirect_stdout

def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    while handling custom layers and standard ones.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for name, layer in model.named_children():
        if isinstance(layer, torch.nn.ModuleList) or isinstance(layer, torch.nn.Sequential):
            # If the layer is a ModuleList or Sequential, recursively flatten it
            modules.extend(flatten_model(layer))
        elif isinstance(layer, torch.nn.Conv2d):
            # If it's a Conv2d, add Conv2d and BatchNorm if exists
            modules.append(layer)
            # Check if BatchNorm is attached
            if hasattr(layer, 'norm'):
                modules.append(layer.norm)
            # Check for activation function (e.g., SiLU or ReLU)
            if hasattr(layer, 'relu'):
                modules.append(layer.relu)
        elif isinstance(layer, torch.nn.BatchNorm2d):
            modules.append(layer)
        elif isinstance(layer, torch.nn.ReLU) or isinstance(layer, torch.nn.SiLU):
            modules.append(layer)
        elif isinstance(layer, torch.nn.MaxPool2d):
            modules.append(layer)
        else:
            # For any other layer, just append it directly
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model
        modules = flatten_model(model)
        
        # Now the backbone is clearly a flat Sequential model of primitive layers
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)


# Usage Example
if __name__ == "__main__":
    from nets import tinysimov3

    # Get the YOLO model
    model = tinysimov3.yolo_v8()
    model.fuse()  # Critical step for weight fusion!
    model.eval()
    
    # Create a generalized wrapper for any model
    generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

    # Save model summary to a text file
    with open('model_summary_fused.txt', 'w') as f:
        with redirect_stdout(f):
            summary(generalized_model, (3, 640, 640))


AttributeError: module 'nets.tinysimov3' has no attribute 'yolo_v8_bn'

In [ ]:
import torch
from torchsummary import summary
from nets import tinysimov2

model = tinysimov2.yolo_v8_s()

In [ ]:
print(model)

In [1]:
from nets import samnet

In [2]:
model = samnet.sam_yolo_v8_s(20, img_size=(1024,1024)).cuda()  # Changed
    

/home/mdi220/anaconda3/envs/YOLO/lib/python3.10/site-packages/segment_anything/build_sam.py:105: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(f)


In [3]:
print(model)

SAMYOLO(
  (sam_encoder): ImageEncoderViT(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    )
    (blocks): ModuleList(
      (0-11): 12 x Block(
        (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (attn): Attention(
          (qkv): Linear(in_features=768, out_features=2304, bias=True)
          (proj): Linear(in_features=768, out_features=768, bias=True)
        )
        (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): Linear(in_features=3072, out_features=768, bias=True)
          (act): GELU(approximate='none')
        )
      )
    )
    (neck): Sequential(
      (0): Conv2d(768, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): LayerNorm2d()
      (2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (3): LayerNorm2d()
    

In [5]:
import numpy
import torch
import torchvision
from torch.nn.functional import cross_entropy, one_hot

import torch
from torchsummary import summary
from contextlib import redirect_stdout


def flatten_model(model):
    """
    This function flattens a model into a Sequential-like list of layers
    by recursively processing all child modules.
    """
    modules = []
    
    # Iterate through all child modules in the model
    for _, layer in model.named_children():
        # If the layer has children, recursively flatten them
        if list(layer.children()):
            modules.extend(flatten_model(layer))
        else:
            # Add the leaf layer
            modules.append(layer)
    
    return modules

class GeneralizedBackboneWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        # Flatten the model into a list of primitive layers
        modules = flatten_model(model)
        
        # Wrap the flattened layers into a Sequential backbone
        self.backbone = torch.nn.Sequential(*modules)

    def forward(self, x):
        return self.backbone(x)

In [7]:
model.eval()

# Create a generalized wrapper for any model
generalized_model = GeneralizedBackboneWrapper(model).cuda()

# Save model summary to a text file
with open('model_summary_fused.txt', 'w') as f:
             with redirect_stdout(f):
                          summary(generalized_model, (3, 1024, 1024))

RuntimeError: Given normalized_shape=[768], expected input with shape [*, 768], but got input of size[2, 768, 64, 64]

In [ ]:
from nets import tinysimov3

# Get the YOLO model
model = tinysimov3.yolo_v8()
model.fuse()  # Critical step for weight fusion!
model.eval()

# Create a generalized wrapper for any model
generalized_model = GeneralizedBackboneWrapper(model.net).cuda()

# Save model summary to a text file
with open('model_summary_fused.txt', 'w') as f:
             with redirect_stdout(f):
                          summary(generalized_model, (3, 640, 640))